In [1]:
import os
import time
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import rasterio

In [2]:

# THIS FUNCTION IS GOOD IF YOU RUN IT ON THE SAME INPUT FOLDE3R AS OUTPUT FOLDER, OTHERWISE IT NEEDS FIX!
def process_file(input_path, output_path):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with rasterio.open(input_path) as src:
        if src.count >= 4:
            rgb_data = src.read([1, 2, 3])
            profile = src.profile
            profile.update(count=3)
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(rgb_data)
            return f"Saved: {output_path}"
        else:
            return f"Skipped: {input_path}, only {src.count} bands."

def remove_nir_band(input_dir, output_dir, max_workers=4):
    start_time = time.time()
    os.makedirs(output_dir, exist_ok=True)
    files = [f for f in os.listdir(input_dir) if f.lower().endswith(".tif")]

    args = [(os.path.join(input_dir, f), os.path.join(output_dir, f))
            for f in files if not os.path.exists(os.path.join(output_dir, f))]

    if not args:
        print("All files already processed.")
        return

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for res in tqdm(executor.map(lambda p: process_file(*p), args),
                        total=len(args), desc="Processing rasters"):
            results.append(res)

    end_time = time.time()
    for r in results:
        print(r)
    print(f"\nFinished processing {len(args)} files with {max_workers} workers in {end_time - start_time:.2f} seconds.")

In [3]:
remove_nir_band(input_dir="/home/jovyan/work/imagery/Nepal/nepal_27112025/clean", 
                output_dir="/home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb", 
                max_workers=8)

Processing rasters: 100%|██████████| 179/179 [1:20:23<00:00, 26.95s/it]

Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25APR04050936-S3DS_R1C1-200010219280_01_P001.tif
Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25APR04050936-S3DS_R2C1-200010219280_01_P001.tif
Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25OCT10052134-S3DS_R6C1-200010241847_01_P001.tif
Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25OCT10052134-S3DS_R6C2-200010241847_01_P001.tif
Skipped: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean/25APR04050936-S3DS_R1C2-200010219280_01_P001.tif, only 3 bands.
Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25FEB19051452-S3DS_R4C2-200010241866_01_P001.tif
Skipped: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean/25APR04050936-S3DS_R3C2-200010219280_01_P001.tif, only 3 bands.
Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25APR04050936-S3DS_R3C1-200010219280_01_P001.tif
Saved: /home/jovyan/work/imagery/Nepal/nepal_27112025/clean_rgb/25APR0

In [2]:
# from concurrent.futures import ThreadPoolExecutor
# from tqdm import tqdm
# import os, rasterio

# def process_file(input_path, output_path):
#     os.makedirs(os.path.dirname(output_path), exist_ok=True)
#     with rasterio.open(input_path) as src:
#         if src.count >= 4:
#             rgb_data = src.read([1, 2, 3])
#             profile = src.profile
#             profile.update(count=3)
#             with rasterio.open(output_path, "w", **profile) as dst:
#                 dst.write(rgb_data)
#         else:
#             print(f"Skipped: {input_path}, only {src.count} bands.")

# def remove_nir_band(input_dir, output_dir, max_workers=4):
#     files = os.listdir(input_dir)
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         list(tqdm(
#             executor.map(
#                 lambda f: process_file(
#                     os.path.join(input_dir, f),
#                     os.path.join(output_dir, f)
#                 ),
#                 files
#             ),
#             total=len(files),
#             desc="Processing rasters"
#         ))


In [ ]:
remove_nir_band(input_dir="/home/jovyan/work/imagery/afghanistan/south_afg/batch1_unzipped", 
                output_dir="/home/jovyan/work/imagery/afghanistan/south_afg/batch_1_rgb", 
                max_workers=8)

In [5]:
import os
os.cpu_count()

28

In [1]:
import geopandas as gpd

gdf = gpd.read_file("/home/jovyan/work/predictions/afghanistan/south_afghanistan/B7_2025/batch4/geojson/predicted_buildings.geojson")

gdf.to_file("/home/jovyan/work/predictions/afghanistan/south_afghanistan/B7_2025/batch4/geojson/batch_4.gpkg")

/opt/conda/lib/python3.9/site-packages/geopandas/io/file.py:362: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  pd.Int64Index,
